
# تحلیل میزان رضایت کاربران از گارانتی محصولات


هدف: شناسایی نظراتی که درباره گارانتی/ضمانت محصول صحبت می‌کنند و محاسبه میانگین امتیاز overall برای هر محصول.

نکته روش‌شناختی: ستون overall امتیاز کلی محصول است، نه امتیاز مستقیم گارانتی. بنابراین خروجی، تخمینی از رضایت در میان نظراتی است که به گارانتی اشاره کرده‌اند.

#  نصب کتابخانه‌ها و آماده‌سازی محیط

In [1]:
!pip -q install gdown rapidfuzz plotly

import os
import re
import html
import gdown
import numpy as np
import pandas as pd
import plotly.express as px
from collections import Counter
from rapidfuzz.fuzz import ratio as fuzzy_ratio
from IPython.display import display
pd.set_option("display.max_colwidth", 180)

#  دریافت داده‌ها و تعیین تنظیمات اولیه


In [ ]:
TRAIN_PATH= "/content/train.csv"
BRAND_PATH= "/content/brand.csv"

if not os.path.exists(TRAIN_PATH):
    gdown.download(
        "https://drive.google.com/uc?id=1-AlW7oNJHaqi3xk_9dWHUS52Dzl_FmFW",
        TRAIN_PATH, quiet=False)

if not os.path.exists(BRAND_PATH):
    gdown.download(
        "https://drive.google.com/uc?id=1I9aPAvvYgQWdHGKtnd7IeTGXpx8vOm4h",
        BRAND_PATH, quiet=False)

CHUNK_SIZE= 50_000
MIN_REVIEWS= 3
RANDOM_STATE= 42
USE_SUMMARY= True

USECOLS= ["asin", "overall", "reviewText", "summary"]

# تعریف کلمات اصلی برای شناسایی نظرات مرتبط با گارانتی


In [ ]:
TOKEN_RE= re.compile(r"[a-z]+")

SEED_WORDS= ["warranty", "guarantee"]

print("Seed words:", SEED_WORDS)
print("FastText training skipped!")

Seed words: ['warranty', 'guarantee']
FastText training skipped!


# شناسایی غلط‌های املایی احتمالی

In [ ]:
vocab= Counter()

for chunk in pd.read_csv(TRAIN_PATH, usecols= ["reviewText", "summary"], dtype= {"reviewText": "string", "summary": "string"}, chunksize= CHUNK_SIZE):

    if USE_SUMMARY:
        texts= (chunk["reviewText"].fillna("") + " " + chunk["summary"].fillna(""))
    else:
        texts= chunk["reviewText"].fillna("")

    for text in texts:
        vocab.update(TOKEN_RE.findall(str(text).lower()))

print("تعداد واژه‌های یکتای شمارش‌شده:", len(vocab))

تعداد واژه‌های یکتای شمارش‌شده: 190555


In [ ]:
TYPO_MIN_COUNT= 3
TYPO_MIN_LENGTH= 6
TYPO_SIMILARITY_THRESHOLD= 80

typo_rows= []

for word, count in vocab.items():
    if count < TYPO_MIN_COUNT or len(word) < TYPO_MIN_LENGTH:
        continue

    if min(abs(len(word) - len(seed)) for seed in SEED_WORDS) > 3:
        continue

    similarity= max(
        fuzzy_ratio(word, seed)
        for seed in SEED_WORDS)

    if similarity >= TYPO_SIMILARITY_THRESHOLD and word not in SEED_WORDS:
        typo_rows.append((word, count, round(similarity / 100, 3)))

typo_df= pd.DataFrame(typo_rows, columns =["word", "count", "similarity"])

if not typo_df.empty:
    typo_df= typo_df.sort_values(["similarity", "count"], ascending= [False, False]).reset_index(drop= True)

display(typo_df.head(60))

,word,count,similarity
0,guaranteed,693,0.947
1,guarantees,155,0.947
2,guarrantee,4,0.947
3,warrantly,6,0.941
4,gurantee,5,0.941
5,warrranty,3,0.941
6,warrant,463,0.933
7,waranty,39,0.933
8,warrany,12,0.933
9,warraty,5,0.933


# تأیید واژگان معنایی و غلط‌های املایی

In [ ]:
# واژه‌های مرتبط با گارانتی و ضمانت
APPROVED_SEMANTIC_TERMS= [
    "warrantee",
    "warrantees",
    "warranties",
    "warrantied",
    "warrants",
    "guaranteed",
    "guarantees",
    "guaranty",
    "guaranties",
    "guaranteeing"
]

# غلط‌های املایی پرتکرار
APPROVED_TYPO_TERMS= [
    "guarrantee",
    "warrantly",
    "gurantee",
    "warrranty",
    "waranty",
    "warrany",
    "warraty",
    "guarentee",
    "gaurantee",
    "guarnteed",
    "warrenty",
    "watranty",
    "guarenteed",
    "gaurantees",
    "gauranteed",
    "warantee",
    "warrently",
    "warenty",
    "warrent"
]

BASE_TERMS= [
    "warranty",
    "warranties",
    "warrantee",
    "warrantees",
    "guarantee",
    "guarantees",
    "guaranteed",
    "guaranteeing",
    "guaranty",
    "guaranties"
]

warranty_terms= sorted(set(BASE_TERMS + APPROVED_SEMANTIC_TERMS + APPROVED_TYPO_TERMS), key= len, reverse= True)

print("تعداد واژه‌های نهایی:", len(warranty_terms))
print(warranty_terms)

تعداد واژه‌های نهایی: 31
['guaranteeing', 'warrantied', 'guarrantee', 'warranties', 'gauranteed', 'guarenteed', 'warrantees', 'guaranteed', 'guaranties', 'gaurantees', 'guarantees', 'guarnteed', 'guarentee', 'guarantee', 'warrently', 'warrranty', 'warrantly', 'warrantee', 'gaurantee', 'gurantee', 'warranty', 'warrenty', 'warantee', 'watranty', 'guaranty', 'warrants', 'warrany', 'warenty', 'warrent', 'warraty', 'waranty']


# ساخت الگوی جست‌وجو

In [ ]:
pattern= re.compile(
    r"(?<![a-z])(?:"
    + "|".join(re.escape(term) for term in warranty_terms)
    + r")(?![a-z])",
    flags= re.IGNORECASE
)

def find_warranty_matches(text):
    if pd.isna(text):
        return []
    return pattern.findall(str(text))

# استخراج نظرات مرتبط از کل داده

In [ ]:
parts= []

n_total= 0
n_valid_scores= 0
sum_valid_scores= 0.0

for chunk_no, chunk in enumerate(
    pd.read_csv(
        TRAIN_PATH,
        usecols= USECOLS,
        dtype= {
            "asin": "string",
            "reviewText": "string",
            "summary": "string"
        },
        chunksize= CHUNK_SIZE
    )
):

    n_total += len(chunk)
    chunk["overall"]= pd.to_numeric(chunk["overall"], errors= "coerce")
    valid_scores= chunk["overall"].dropna()
    n_valid_scores += len(valid_scores)
    sum_valid_scores += valid_scores.sum()

    if USE_SUMMARY:
        combined_text= (
            chunk["reviewText"].fillna("") + " " +
            chunk["summary"].fillna("")
        )
    else:
        combined_text= chunk["reviewText"].fillna("")

    matches= combined_text.map(find_warranty_matches)

    mask= (
        matches.map(bool)
        & chunk["asin"].notna()
        & chunk["overall"].notna()
    )

    if mask.any():

        selected= chunk.loc[
            mask,
            ["asin", "overall", "reviewText", "summary"]
        ].copy()

        selected["matched_terms"]= matches.loc[mask].map(
            lambda items: ", ".join(
                sorted(set(x.lower() for x in items))
            )
        )

        parts.append(selected)

    if (chunk_no + 1) % 10 == 0:
        print(f"Processed {n_total:,} reviews")

if parts:
    final_hits= pd.concat(parts, ignore_index=True)
else:
    final_hits= pd.DataFrame(columns= ["asin", "overall", "reviewText", "summary", "matched_terms"])

avg_all_reviews= (sum_valid_scores / n_valid_scores
    if n_valid_scores else np.nan)

avg_warranty_reviews= (final_hits["overall"].mean()
    if not final_hits.empty else np.nan)

print("تعداد کل نظرات:", f"{n_total:,}")
print("تعداد نظرات دارای امتیاز معتبر:", f"{n_valid_scores:,}")
print("تعداد نظرات شناسایی‌شده درباره گارانتی:", f"{len(final_hits):,}")
print("میانگین امتیاز کل نظرات:", round(avg_all_reviews, 3))
print("میانگین امتیاز نظرات مرتبط با گارانتی:", round(avg_warranty_reviews, 3))

Processed 500,000 reviews
تعداد کل نظرات: 838,944
تعداد نظرات دارای امتیاز معتبر: 838,944
تعداد نظرات شناسایی‌شده درباره گارانتی: 18,805
میانگین امتیاز کل نظرات: 4.021
میانگین امتیاز نظرات مرتبط با گارانتی: 3.201


In [ ]:
if not final_hits.empty:
    sample_n= min(30, len(final_hits))
    display(final_hits[
            ["asin", "overall", "matched_terms", "reviewText", "summary"]
        ].sample(sample_n, random_state= RANDOM_STATE))
else:
    print("هیچ نظر مرتبطی پیدا نشد؛ مسیر داده و فهرست واژگان را بررسی کنید.")

,asin,overall,matched_terms,reviewText,summary
9728,B00WZ1IFPE,5,warranty,"replaced same unit for DVR security system (NightOwl from Costco). For whatever reason, both the DVR power supply and Camera power supply failed just a few weeks from each othe...",Hope it works longer than original
2360,B006VHF9TI,5,warranty,"Great product, easy to use and the on-line tracker architecture is very good. Customer service is AWESOME--at 18 months of use, my unit would could not be recognized and their...","Great product, easy to use and the on-line tracker ..."
12020,B015E9Z87Q,1,warranty,"I bought the refurb model. Due to being busy, I didn't get to really go to try this until I was in a hotel room on a business trip about 1.5 months after I bought it. It was a ...","Only takes very latest software versions, and still never worked for me"
14962,B01DNE7H1O,5,warranty,"TL;DR Everything works as you would expect, its not going to blow you away, but if you're looking for a budget keyboard case that registers every keystroke, has backlighting an...",The Price is Right
11292,B013HSW4N2,1,warranty,Wow. What a disappointment. 2nd pair I have had and this 2nd one just suddenly stopped working out of nowhere. I tried to get an RMA for these headphones from Jaybird themselve...,Disappointed
9776,B00X5P8BJO,5,warrantee,Had to return...wouldn't hold a charge. Update- Arteck reached out and volunteered to me that they had a two year warrantee and subsequently sent a new replacement. I wish e...,Wouldn't hold a charge
42,B00004Z7G9,4,warranty,I go through headphones quite a bit and I was getting tired of cheap cans with wire fray. I also own a few 1960s amplifiers (tube and solid-state) and I wanted to pair them wi...,"Great headphones, a bit sensitive to revealing flaws in old equipment."
928,B001DAT0XE,4,warranty,"While the disc has cracked enough to be unusable at this point, it has done so after almost six years on a dashboard in Texas with no respite. The adhesive is still strong, too...",While the disc has cracked enough to be unusable at ...
18563,B01B25PX30,1,warranty,"Looks very nice, but we had the same issue others report about the plastic piece breaking. It was just out of warranty, too. I contacted support and there is no way to repair i...","DO NOT BUY, bad design"
13806,B01AAKZRP2,5,warranty,"It's interesting to me that they're splitting the EVO line. Now there is the lower end, this 750 EVO and then there's the higher end which is 850 EVO. This 750 EVO has a little...",The 750 EVO is a good budget SSD


# ارزیابی خودکار روش استخراج

In [ ]:
# واژگان اصلی و قطعی
CORE_TERMS= [
    "warranty",
    "warranties",
    "warrantee",
    "warrantees",
    "guarantee",
    "guarantees",
    "guaranteed",
    "guaranteeing",
    "guaranty",
    "guaranties"
]

core_pattern= re.compile(
    r"(?<![a-z])(?:"
    + "|".join(re.escape(term) for term in CORE_TERMS)
    + r")(?![a-z])",
    flags= re.IGNORECASE
)

# متن نظرات استخراج‌شده
if not final_hits.empty:

    if USE_SUMMARY:
        evaluation_text= (
            final_hits["reviewText"].fillna("") + " " +
            final_hits["summary"].fillna("")
        )
    else:
        evaluation_text = final_hits["reviewText"].fillna("")

    # نظراتی که حداقل یک واژه اصلی دارند
    core_matches= evaluation_text.map(
        lambda text: bool(core_pattern.search(str(text)))
    )

    total_extracted= len(final_hits)

    core_count= int(core_matches.sum())

    expanded_only= total_extracted - core_count

    core_coverage= (core_count / total_extracted * 100
        if total_extracted else 0)

    expanded_share = (expanded_only / total_extracted * 100
        if total_extracted else 0)

    evaluation_results= pd.DataFrame({
        "Metric": [
            "Total extracted reviews",
            "Reviews containing core terms",
            "Reviews detected only by expanded terms",
            "Core term coverage (%)",
            "Expanded term contribution (%)"
        ],

        "Value": [
            total_extracted,
            core_count,
            expanded_only,
            round(core_coverage, 2),
            round(expanded_share, 2)
        ]
    })

    display(evaluation_results)

else:

    print("No warranty-related reviews found.")

,Metric,Value
0,Total extracted reviews,18805.00
1,Reviews containing core terms,18456.00
2,Reviews detected only by expanded terms,349.00
3,Core term coverage (%),98.14
4,Expanded term contribution (%),1.86


# محاسبه میانگین امتیاز برای هر محصول

In [ ]:
if final_hits.empty:

    product_result= pd.DataFrame(
        columns= ["asin", "avg_warranty_rating", "n_reviews"]
    )

else:

    product_result= (
        final_hits.groupby("asin", as_index=False)
        .agg(
            avg_warranty_rating=("overall", "mean"),
            n_reviews=("overall", "count")
        )
    )

brand_raw= pd.read_csv(BRAND_PATH, dtype="string")

brand_raw.columns= (brand_raw.columns.str.strip().str.lower())

if "asin" not in brand_raw.columns:

    raise ValueError(
        "ستون asin در فایل نگاشت پیدا نشد. ستون‌های موجود: "
        + str(brand_raw.columns.tolist())
    )

metadata_cols= ["asin"] + [
    col for col in ["title", "brand"]
    if col in brand_raw.columns]

brand_map= brand_raw[metadata_cols].copy()


def first_non_empty(series):

    values= series.dropna().astype(str).str.strip()
    values= values[values.ne("")]

    return values.iloc[0] if len(values) else pd.NA


agg_dict= {
    col: first_non_empty
    for col in metadata_cols
    if col != "asin"}

if agg_dict:
    brand_map= brand_map.groupby("asin", as_index= False).agg(agg_dict)

else:
    brand_map \= brand_map.drop_duplicates("asin")


if "title" in brand_map.columns:
    brand_map["title"]= brand_map["title"].map(
        lambda x: html.unescape(x)
        if isinstance(x, str) else x)


product_result= product_result.merge(brand_map, on= "asin", how= "left", validate= "one_to_one")

print("تعداد محصولات دارای نظر مرتبط:", len(product_result))

for k in [1, 2, 3, 5, 10]:

    print(f"محصولات دارای حداقل {k} نظر مرتبط:", int((product_result["n_reviews"] >= k).sum()))

تعداد محصولات دارای نظر مرتبط: 10175
محصولات دارای حداقل 1 نظر مرتبط: 10175
محصولات دارای حداقل 2 نظر مرتبط: 3128
محصولات دارای حداقل 3 نظر مرتبط: 1548
محصولات دارای حداقل 5 نظر مرتبط: 601
محصولات دارای حداقل 10 نظر مرتبط: 179


# فیلتر محصولات و نمایش نتایج

In [ ]:
result= (product_result[product_result["n_reviews"] >= MIN_REVIEWS].sort_values(["avg_warranty_rating", "n_reviews"], ascending= [False, False]).reset_index(drop= True))

result["avg_warranty_rating"]= result["avg_warranty_rating"].round(3)

print("۱۵ محصول با بالاترین میانگین امتیاز:")
display(result.head(15))

lowest_result= (product_result[product_result["n_reviews"] >= MIN_REVIEWS].sort_values(["avg_warranty_rating", "n_reviews"], ascending= [True, False]).reset_index(drop= True))

lowest_result["avg_warranty_rating"]= lowest_result["avg_warranty_rating"].round(3)

print("۱۵ محصول با پایین‌ترین میانگین امتیاز:")
display(lowest_result.head(15))

۱۵ محصول با بالاترین میانگین امتیاز:


,asin,avg_warranty_rating,n_reviews,title,brand
0,B00N1O1NQW,5.0,29,"iPhone Charger Syncwire Lightning Cable - [Apple MFi Certified] - 3.3ft/1m for iPhone Xs, Xs Max, XR, X, 8, 8 Plus, 7, 7 Plus, 6s, 6s Plus, 6, 6 Plus, SE, 5s, 5c, 5, iPad Mini/...",Syncwire
1,B00EKJQM5E,5.0,9,"EVGA SuperNOVA 1000 P2, 80+ PLATINUM 1000W, Fully Modular, EVGA ECO Mode, 10 Year Warranty, Includes FREE Power On Self Tester, Power Supply 220-P2-1000-XR",EVGA
2,B00650P2ZC,5.0,7,"Noctua NF-F12 PWM, 4-Pin Premium Quiet Fan (120mm, Brown)",Noctua
3,B015PC7R6W,5.0,7,"Syncwire Lightning to USB A Cable - Nylon Braided [Apple Mfi Certified] iPhone Charger for iPhone Xs, Max, XR, X, 8, Plus, 7, Plus, 6S, Plus, 6, Plus, iPad Mini/Air/Pro & More ...",Syncwire
4,B0002SKHIK,5.0,6,Pelican 1510 Case With Foam (Black),Pelican
5,B000WU2LXC,5.0,6,"ARCTIC MX-2 - Thermal Compound Paste, Carbon Based High Performance, Heatsink Paste, Thermal Compound CPU for All Coolers, Thermal Interface Material - 4 Grams",ARCTIC
6,B00PK00WRG,5.0,6,"82mm X2 UV Filter For Camera Lenses - UV Protection Photography Filter with Lens Cloth - MRC8, Nanotech Coatings, Ultra-slim, Traction Frame, Weather-Sealed by Breakthrough Pho...",Breakthrough Photography
7,B00EU799O2,5.0,5,"Macbook Pro 15 Sleeve, Snugg - Brown Leather Sleeve Case Protective Cover for Macbook Pro 15",Snugg
8,B00O0KISQE,5.0,5,Anker USB 3.0 4-Port Portable Aluminum Hub with 2-Foot USB 3.0 Cable (Carbon),Anker
9,B00RNGOBYK,5.0,5,"amFilm iPad 9.7 Screen Protector Tempered Glass for Apple iPad 9.7"" (2018, 2017)/iPad Pro 9.7 Inch/iPad Air/iPad Air 2, New iPad 5th, 6th Gen, Apple Pencil Compatible",amFilm


۱۵ محصول با پایین‌ترین میانگین امتیاز:


,asin,avg_warranty_rating,n_reviews,title,brand
0,B019GZ53VU,1.0,7,VIZIO 32 inches 720 p LED TV D32HN-E0 (2016),VIZIO
1,B008FC8FJ8,1.0,6,ViewSonic VFD820-50 8-Inch Digital Photo Frame (Black),ViewSonic
2,B0111MRL4S,1.0,6,"NETGEAR Nighthawk AC1900 VDSL/ADSL Modem Router Certified with CenturyLink - Non-bonded, DSL Internet Only (D7000)",NETGEAR
3,B01606KJ6C,1.0,6,"Microsoft Surface Pro 4 (128 GB, 4 GB RAM, Intel Core i5)",Microsoft
4,B00O8B7CP8,1.0,5,"Pioneer AVH-X3700BHS DVD Receiver with 6.2-Inch Display, Bluetooth, Siri Eyes Free, SiriusXM-Ready, HD Radio, Android Music Support, Pandora and Dual Camera Inputs",Pioneer
5,B00WWI9UW2,1.0,5,ViewSonic VFD820-70 8-Inch Digital Photo Frame (White),ViewSonic
6,B0010VGC6Y,1.0,4,Linkskey 2-Port Dual Monitor DVI/DVI USB KVM + 7.1 Surround/Microphone/USB with Cables (LDV-DM702AUSK),Linkskey
7,B0012Z3MKW,1.0,4,Thermaltake BlacX 2.5”/3.5” Sata I/II/III USB 2.0 External Hard Drive Enclosure Docking Station N0028USU,Thermaltake
8,B008DVMRXG,1.0,4,Mediabridge USB 2.0 - Micro-USB to USB Coiled Cable (1'-3') - High-Speed A Male to Micro B - Black,Mediabridge
9,B00I8O6OQ4,1.0,4,(Old Model) Seagate 2TB Laptop HDD SATA III 2.5-Inch Internal Bare Drive 9.5MM (ST2000LM003),Seagate


# مقایسه و مصورسازی نتایج


In [ ]:
comparison = pd.DataFrame({"نوع نظر": ["تمام نظرات دارای امتیاز", "نظرات مرتبط با گارانتی"], "میانگین امتیاز": [avg_all_reviews, avg_warranty_reviews]})

display(comparison.round(3))

fig= px.bar(comparison, x= "نوع نظر", y= "میانگین امتیاز", text= "میانگین امتیاز", range_y= [0, 5], title= "مقایسه میانگین امتیاز کل و نظرات مرتبط با گارانتی")
fig.show()

if not final_hits.empty:
    score_dist= (
        final_hits["overall"]
        .value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(1)
        .rename_axis("امتیاز")
        .reset_index(name= "درصد"))

    fig= px.bar(
        score_dist,
        x= "امتیاز",
        y= "درصد",
        text= "درصد",
        title= "توزیع امتیاز نظرات مرتبط با گارانتی")
    fig.show()

if not product_result.empty:
    fig= px.scatter(
        product_result[product_result["n_reviews"] >= MIN_REVIEWS],
        x= "n_reviews",
        y= "avg_warranty_rating",
        hover_data= ["asin"],
        log_x= True,
        title= "تعداد نظرات مرتبط در برابر میانگین امتیاز محصول")
    fig.show()

,نوع نظر,میانگین امتیاز
0,تمام نظرات دارای امتیاز,4.021
1,نظرات مرتبط با گارانتی,3.201


#  ذخیره‌سازی خروجی‌ها

In [ ]:
result.to_csv("warranty_avg_rating.csv", index= False)
product_result.to_csv("warranty_avg_rating_all_products.csv", index= False)
final_hits.to_csv("warranty_related_reviews.csv", index= False)
evaluation_results.to_csv("warranty_automatic_evaluation.csv", index= False)
print("All output files saved successfully!")

All output files saved successfully!
